1.CSV se directly load karna (pd.read_csv) — dtype specify karna na bhoolna (Day 3 wala lesson yaad hai):

In [2]:
import pandas as pd

df_csv = pd.read_csv(
    '../data/raw/online_retail_II.csv',
    dtype={'Invoice': str, 'StockCode': str}
)
print(df_csv.shape)
print(df_csv.dtypes)

(1067371, 8)
Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object


2.read_csv() ke useful parameters — bade files ke liye kaam aate hain:

In [4]:
# Sirf specific columns load karna (memory bachane ke liye)
df_partial = pd.read_csv(
    '../data/raw/online_retail_II.csv',
    dtype={'Invoice': str, 'StockCode': str},
    usecols=['Invoice', 'Customer ID', 'Country', 'InvoiceDate']
)
print(df_partial.shape)
print(df_partial.columns)

# Sirf pehle N rows load karna (testing ke liye fast)
df_sample = pd.read_csv('../data/raw/online_retail_II.csv', dtype={'Invoice': str}, nrows=1000)
print(df_sample.shape)

# InvoiceDate ko directly datetime bana kar load karna
df_dates = pd.read_csv(
    '../data/raw/online_retail_II.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate']
)
print(df_dates['InvoiceDate'].dtype)   # datetime64 aana chahiye, string nahi

(1067371, 4)
Index(['Invoice', 'InvoiceDate', 'Customer ID', 'Country'], dtype='str')
(1000, 8)
datetime64[us]


3.SQL database se load karna (jo tumhare paas already SQLite mein hai):

In [5]:
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

# Poori table load karna
df_transactions = pd.read_sql("SELECT * FROM transactions", conn)
print(df_transactions.shape)

# Specific query se load karna (jaisa Day 1-19 mein karte aaye ho)
df_uk_only = pd.read_sql("SELECT * FROM transactions WHERE Country = 'United Kingdom'", conn)
print(df_uk_only.shape)

conn.close()

(1067371, 8)
(981330, 8)


4.Excel se load karna (agar kabhi zaroorat pade — good to know):

In [6]:
# Agar tumhare paas Excel version bhi hota:
# df_excel = pd.read_excel('data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')

# Multiple sheets ek saath dictionary mein:
# all_sheets = pd.read_excel('data/raw/online_retail_II.xlsx', sheet_name=None)

(Humhara data CSV mein hai to yeh sirf reference ke liye — agar kabhi Excel file milti hai to yeh syntax kaam aayega)

5.CSV vs SQL load karne mein farak — kab kya use karo:

In [7]:
import time

# CSV load time
start = time.time()
df_from_csv = pd.read_csv('../data/raw/online_retail_II.csv', dtype={'Invoice': str, 'StockCode': str})
print(f"CSV load time: {time.time() - start:.2f}s")

# SQL load time
conn = sqlite3.connect('../data/db/ecommerce.db')
start = time.time()
df_from_sql = pd.read_sql("SELECT * FROM transactions", conn)
print(f"SQL load time: {time.time() - start:.2f}s")
conn.close()

CSV load time: 0.85s
SQL load time: 2.07s


Note :- SQL se load karna tab better hai jab humhe filtered/aggregated data chahiye (query database mein hi filter kar deti hai, kam data Python mein aata hai). Pura raw data chahiye to CSV bhi theek hai.

Practice questions

1.pd.read_sql() se ek query likho jo sirf Germany ke transactions load kare, aur compare karo iska .shape poore data ke .shape se.

In [15]:

# Database connection establish karein (example ke liye SQLite)
conn = sqlite3.connect('../data/db/ecommerce.db')

# 1. Sirf Germany ke transactions load karne ki query
query_germany = "SELECT * FROM transactions WHERE Country = 'Germany'"
df_germany = pd.read_sql(query_germany, conn)

# 2. Poora data load karne ki query (comparison ke liye)
query_full = 'SELECT * FROM transactions'
df_full = pd.read_sql(query_full, conn)

# 3. Shape compare karein
print('Germany Data Shape:', df_germany.shape)
print('Poora Data Shape:', df_full.shape)

conn.close()

Germany Data Shape: (17624, 8)
Poora Data Shape: (1067371, 8)


2.usecols parameter use karke sirf StockCode, Description, aur Price columns load karo CSV se — dekho poore file se kitna kam memory use hota hai (df.memory_usage(deep=True).sum() se check kar sakte ho).

In [16]:
file_path = '../data/raw/online_retail_II.csv'

# A. Poori CSV file sabhi columns ke sath load karein
df_full_csv = pd.read_csv(file_path)

# B. Sirf specific columns load karein usecols use karke
cols = ['StockCode', 'Description', 'Price']
df_subset_csv = pd.read_csv(file_path, usecols=cols)

# C. Memory usage check karein (in MB)
mem_full = df_full_csv.memory_usage(deep=True).sum() / (1024 * 1024)
mem_subset = df_subset_csv.memory_usage(deep=True).sum() / (1024 * 1024)

print(f'Poore Data ki Memory: {mem_full:.2f} MB')
print(f'Subset Data ki Memory: {mem_subset:.2f} MB')
print(f'Total Memory Saved: {mem_full - mem_subset:.2f} MB')

Poore Data ki Memory: 345.12 MB
Subset Data ki Memory: 140.08 MB
Total Memory Saved: 205.04 MB
